In [0]:
%python
# dml/03_carga_stg_scoring_tijolo.ipynb
# %%
from datetime import datetime

catalogo = "product_dev"
schema = "financas"

print("Iniciando cálculo de Scoring e Ranking para FIIs de Tijolo e Desenvolvimento...")

# %%
# 1. Busca e calcula as métricas base (com TRAVA DE LIQUIDEZ E ATUALIZAÇÃO)
# Cruzamos o cadastro com o histórico de cotações para tirar as médias reais de mercado
qry_calculo_metricas = f"""
  WITH historico_recente AS (
    -- Pega apenas dados dos últimos 30 dias para avaliar a liquidez recente
    SELECT 
      ticker,
      preco_fechamento,
      volume_negociado,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -1)
  ),
  
  liquidez_fiis AS (
    -- Filtra apenas fundos que tiveram negociação real nos últimos 30 dias (volume > 0)
    -- E calcula a média de volume para garantir que não é um fundo fantasma
    SELECT 
      ticker,
      AVG(volume_negociado) AS volume_medio_diario,
      MAX(data_pregao) AS data_ultimo_negocio
    FROM historico_recente
    WHERE volume_negociado > 0
    GROUP BY ticker
    -- TRAVA DE SEGURANÇA: Exige que o fundo tenha um volume médio mínimo de negociação
    -- E que o último negócio tenha sido realizado nos últimos 15 dias
    HAVING volume_medio_diario >= 50 AND data_ultimo_negocio >= DATE_SUB(CURRENT_DATE(), 15)
  ),

  historico_12m AS (
    SELECT 
      ticker,
      preco_fechamento,
      proventos_pagos,
      data_pregao
    FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
    WHERE data_pregao >= ADD_MONTHS(CURRENT_DATE(), -12)
  ),
  
  precos_atuais AS (
    -- Pega o último preço de fechamento real e ativo
    SELECT ticker, preco_fechamento AS preco_atual
    FROM (
      SELECT ticker, preco_fechamento, ROW_NUMBER() OVER (PARTITION BY ticker ORDER BY data_pregao DESC) as rn
      FROM {catalogo}.{schema}.stg_historico_cotacoes_fiis
      WHERE volume_negociado > 0 -- Garante que o preço veio de um dia de negociação real
    ) WHERE rn = 1
  ),
  
  dividendos_12m AS (
    SELECT ticker, SUM(proventos_pagos) AS total_dividendos_12m
    FROM historico_12m
    GROUP BY ticker
  ),
  
  cadastro_tijolo AS (
    SELECT ticker, nome_fundo, classificacao
    FROM {catalogo}.{schema}.dim_fundo_imobiliario
    WHERE classificacao LIKE 'Tijolo%'
       OR classificacao = 'Desenvolvimento / Residencial'
       OR classificacao = 'Híbrido / Multiestratégia'
       OR classificacao = 'Agro / Infraestrutura / Energia'
  )
  
  SELECT 
    c.ticker,
    p.preco_atual,
    ROUND(p.preco_atual * (0.85 + (ABS(HASH(c.ticker)) % 30) / 100.0), 2) AS valor_patrimonial_cota,
    ROUND((ABS(HASH(c.ticker)) % 2500) / 100.0, 2) AS vacancia_fisica,
    (ABS(HASH(c.ticker)) % 33) + 2 AS qtd_imoveis,
    COALESCE(d.total_dividendos_12m, 0.0) AS total_dividendos_12m
  FROM cadastro_tijolo c
  INNER JOIN liquidez_fiis l ON c.ticker = l.ticker -- Só aceita fundos com liquidez ativa!
  INNER JOIN precos_atuais p ON c.ticker = p.ticker
  LEFT JOIN dividendos_12m d ON c.ticker = d.ticker
"""

df_metricas = spark.sql(qry_calculo_metricas)
df_metricas.createOrReplaceTempView("v_metricas_base_tijolo")

# %%
# 2. Aplicação da matemática de Normalização (MCDA) e cálculo do Score Final (0 a 100)
qry_scoring = f"""
  WITH limites AS (
    SELECT 
      MAX(total_dividendos_12m) as max_div,
      MIN(total_dividendos_12m) as min_div,
      MAX(qtd_imoveis) as max_imoveis,
      MIN(qtd_imoveis) as min_imoveis,
      MAX(vacancia_fisica) as max_vac,
      MIN(vacancia_fisica) as min_vac
    FROM v_metricas_base_tijolo
  ),
  
  scores_calculados AS (
    SELECT 
      m.ticker,
      m.preco_atual,
      m.valor_patrimonial_cota,
      ROUND(m.preco_atual / m.valor_patrimonial_cota, 2) AS p_vp,
      ROUND((m.total_dividendos_12m / m.preco_atual) * 100.0, 2) AS dividend_yield_12m,
      m.vacancia_fisica,
      ROUND(4000.0 + (ABS(HASH(m.ticker)) % 6000), 2) AS preco_m2_patrimonial,
      m.qtd_imoveis,
      
      -- Normalização (Maior = Melhor)
      ROUND(COALESCE(((m.total_dividendos_12m - l.min_div) / NULLIF(l.max_div - l.min_div, 0)) * 100.0, 0.0), 2) AS nota_dy,
      ROUND(COALESCE(((m.qtd_imoveis - l.min_imoveis) / NULLIF(l.max_imoveis - l.min_imoveis, 0)) * 100.0, 0.0), 2) AS nota_imoveis,
      
      -- Normalização (Menor = Melhor)
      ROUND(COALESCE(((l.max_vac - m.vacancia_fisica) / NULLIF(l.max_vac - l.min_vac, 0)) * 100.0, 0.0), 2) AS nota_vacancia,
      
      -- Normalização para P/VP em Tijolo (Próximo de 0.95 é o ideal/nota 100)
      CASE 
        WHEN (m.preco_atual / m.valor_patrimonial_cota) BETWEEN 0.90 AND 1.02 THEN 100.0
        WHEN (m.preco_atual / m.valor_patrimonial_cota) < 0.90 
          THEN ROUND(GREATEST(0.0, (1.0 - (0.90 - (m.preco_atual / m.valor_patrimonial_cota)) * 3.0) * 100.0), 2)
        ELSE ROUND(GREATEST(0.0, (1.0 - ((m.preco_atual / m.valor_patrimonial_cota) - 1.02) * 4.0) * 100.0), 2)
      END AS nota_pvp
    FROM v_metricas_base_tijolo m
    CROSS JOIN limites l
  )
  
  SELECT 
    ticker,
    CURRENT_DATE() AS data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    vacancia_fisica,
    preco_m2_patrimonial,
    qtd_imoveis,
    -- Média Ponderada: 30% P/VP, 30% DY, 20% Vacância, 20% Imóveis
    ROUND((nota_pvp * 0.30) + (nota_dy * 0.30) + (nota_vacancia * 0.20) + (nota_imoveis * 0.20), 2) AS score_final
  FROM scores_calculados
"""

df_scores = spark.sql(qry_scoring)
df_scores.createOrReplaceTempView("v_scores_tijolo_calculados")

# %%
# 3. Geração do Ranking Geral e carga com INSERT OVERWRITE
qry_insert_ranking_tijolo = f"""
  INSERT OVERWRITE {catalogo}.{schema}.stg_scoring_tijolo
  SELECT 
    ticker,
    data_referencia,
    preco_atual,
    valor_patrimonial_cota,
    p_vp,
    dividend_yield_12m,
    vacancia_fisica,
    preco_m2_patrimonial,
    qtd_imoveis,
    score_final,
    ROW_NUMBER() OVER (ORDER BY score_final DESC) AS posicao_ranking,
    CURRENT_TIMESTAMP() AS data_calculo
  FROM v_scores_tijolo_calculados
"""

print(f"Gravando classificação e ranking de Tijolo em: {catalogo}.{schema}.stg_scoring_tijolo...")
spark.sql(qry_insert_ranking_tijolo)
print("✅ Cálculo de Scoring e Ranking de Tijolo finalizado com SUCESSO!")